In [0]:
########################
#Install Dependencies
########################
%pip install openpyxl
%pip install pycountry
# dbutils.library.restartPython()

In [0]:
########################
#Setup for Transformation Tests
########################
this_notebook_state = "decided(a)"

from datetime import datetime, timedelta
run_user = spark.sql("SELECT current_user()").first()[0]
run_tag = "Testing Transformation Tests"
run_by_automation_name = "Transformation_Tests"
run_start_datetime = datetime.now()

dbutils.widgets.text("state_under_test", "")

state_under_test = dbutils.widgets.get("state_under_test")

if state_under_test == "":
    state_under_test = this_notebook_state

#List of Fields to Exclude in Child Notebooks Called
child_fields_to_exclude = ["decisionAndReasonsAvailable", "recordedOutOfTimeDecision"]

print(f"Testing State = {state_under_test}")

from models.test_result import TestResult
from dataclasses import asdict
import os

all_test_results = []
current_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
base_path = current_path.rsplit("/", 1)[0] + "/"
test_results_path = "/Workspace/Users/" + run_user + "/Results/Transformation_Tests"
os.makedirs(test_results_path, exist_ok=True)


In [0]:
#Load Config and Setup Enviorment Variables
M2_bronze = None
M3_bronze = None
M3_silver = None
M6_bronze = None
bat = None
bhoref = None
bac = None
bll = None
b = None
M4_silver = None
M4_bronze = None
docsr = None
H_bronze = None
H_silver = None

config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()
KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"

spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

spark.conf.set(f"fs.azure.account.auth.type.{external_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{external_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{external_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{external_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{external_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
gold_path = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/{state_under_test}"

import json
json_location = dbutils.fs.ls(f"{gold_path}/")[-1]
latest_json_location = json_location.name

#Set Paths
try:
    json_path = f"{gold_path}/{latest_json_location}/JSON/"
    M1_silver = f"{silver_path}/silver_appealcase_detail"
    M1_bronze = f"{bronze_path}/bronze_appealcase_crep_rep_floc_cspon_cfs"
    M2_silver = f"{silver_path}/silver_caseapplicant_detail"
    M3_silver = f"{silver_path}/silver_status_detail"
    C = f"{silver_path}/silver_appealcategory_detail"
    bhc = f"{bronze_path}/bronze_hearing_centres"
    M3_bronze = f"{bronze_path}/bronze_status_htype_clist_list_ltype_court_lsitting_adj"
except:
    print(f"Error during fetch: {str(e)}")

#Create and Load Dataframes
json_data = spark.read.format("json").load(json_path)
M1_silver = spark.read.format("delta").load(M1_silver)
M1_bronze = spark.read.format("delta").load(M1_bronze)
M2_silver = spark.read.format("delta").load(M2_silver)
M3_silver = spark.read.format("delta").load(M3_silver)
C = spark.read.format("delta").load(C)
bhc = spark.read.format("delta").load(bhc)
M3_bronze = spark.read.format("delta").load(M3_bronze)

# Load data that isn't loading
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("appealReferenceNumber", StringType(), True),
    StructField("witness1InterpreterSignLanguage", StringType(), True),
    StructField("witness2InterpreterSignLanguage", StringType(), True),
    StructField("witness3InterpreterSignLanguage", StringType(), True),
    StructField("witness4InterpreterSignLanguage", StringType(), True),
    StructField("witness5InterpreterSignLanguage", StringType(), True),
    StructField("witness6InterpreterSignLanguage", StringType(), True),
    StructField("witness7InterpreterSignLanguage", StringType(), True),
    StructField("witness8InterpreterSignLanguage", StringType(), True),
    StructField("witness9InterpreterSignLanguage", StringType(), True),
    StructField("witness10InterpreterSignLanguage", StringType(), True),
    StructField("witness1InterpreterSpokenLanguage", StringType(), True),
    StructField("witness2InterpreterSpokenLanguage", StringType(), True),
    StructField("witness3InterpreterSpokenLanguage", StringType(), True),
    StructField("witness4InterpreterSpokenLanguage", StringType(), True),
    StructField("witness5InterpreterSpokenLanguage", StringType(), True),
    StructField("witness6InterpreterSpokenLanguage", StringType(), True),
    StructField("witness7InterpreterSpokenLanguage", StringType(), True),
    StructField("witness8InterpreterSpokenLanguage", StringType(), True),
    StructField("witness9InterpreterSpokenLanguage", StringType(), True),
    StructField("witness10InterpreterSpokenLanguage", StringType(), True)
])

print(f"Before - Number of Fields : {str(len(json_data.columns))}")
missing_fields_json = spark.read.schema(schema).json(json_path)
json_data = json_data.join(
    missing_fields_json,
    json_data["appealReferenceNumber"] == missing_fields_json["appealReferenceNumber"],
    "left"
).drop(missing_fields_json["appealReferenceNumber"])
print(f"After - Adding Missing Fields : Count : {str(len(json_data.columns))}")

# Load additional tables needed by chain runners
try: M2_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_appealcase_caseappellant_appellant") if M2_bronze is None else M2_bronze
except: pass
try: M3_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_status_htype_clist_list_ltype_court_lsitting_adj") if M3_bronze is None else M3_bronze
except: pass
try: bat = spark.read.format("delta").load(f"{bronze_path}/bronze_appealtype") if bat is None else bat
except: pass
try: bhoref = spark.read.format("delta").load(f"{bronze_path}/bronze_HORef_cleansing") if bhoref is None else bhoref
except: pass
try: H_silver = spark.read.format("delta").load(f"{silver_path}/silver_history_detail") if H_silver is None else H_silver
except: pass
try: H_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_history") if H_bronze is None else H_bronze
except: pass
try: M4_silver = spark.read.format("delta").load(f"{silver_path}/silver_transaction_detail") if M4_silver is None else M4_silver
except: pass
try: M4_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_appealcase_transaction_transactiontype") if M4_bronze is None else M4_bronze
except: pass
try: M6_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_history") if M6_bronze is None else M6_bronze
except: pass
try: bac = spark.read.format("delta").load(f"{bronze_path}/bronze_appealcategory") if bac is None else bac
except: pass
try: bll = spark.read.format("delta").load(f"{bronze_path}/bronze_listing_location") if bll is None else bll
except: pass
try: docsr = spark.read.format("delta").load(f"{bronze_path}/bronze_documentsreceived") if docsr is None else docsr
except: pass


In [0]:
import time as perf_time
perf_timings = []

# NO_DATA check
record_count = 0 if json_data is None else json_data.count()
if record_count == 0:
    print(f"NO DATA for state: {state_under_test}")
    all_test_results.append(TestResult(
        test_field="",
        status="NO_DATA",
        message=f"No records found for state: {state_under_test}. Tests skipped.",
        test_from_state=state_under_test,
        test_name="data_check"
    ))
else:
    print(f"Found {record_count} records - running tests")

    print(f"Running 11 states...")
    from Test_Functions.paymentPending_appealType_caseData_hearingCentre_flagLabels_general import run as run_pp_chunk1
    from Test_Functions.paymentPending_appellantDetails_legalRepDetails_sponsorDetails import run as run_pp_chunk2
    from Test_Functions.paymentPending_partyIds_payment_remission_homeOffice_defaultMappings import run as run_pp_chunk3
from Test_Functions.paymentPending_Detained import run as run_pp_chunk4
    from Test_Functions.test_helpers import classify_all
    from Test_Functions.run_appealSubmitted import run_all_tests as run_as
    from Test_Functions.run_awaitingRespEvidenceA import run_all_tests as run_areA
    from Test_Functions.run_awaitingRespEvidenceB import run_all_tests as run_areB
    from Test_Functions.run_listing import run_all_tests as run_listing
    from Test_Functions.run_prepareForHearing import run_all_tests as run_pfh
    from Test_Functions.run_decision import run_all_tests as run_decision
    from Test_Functions.run_decidedA import run_all_tests as run_decidedA

    print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
    t0 = perf_time.time()
    res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, state_under_test)
    dur = perf_time.time() - t0
    all_test_results.extend(res)
    perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(res)})
    print(f"  [1/11] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(res):4d} results")

    print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
    t0 = perf_time.time()
    res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, state_under_test)
    dur = perf_time.time() - t0
    all_test_results.extend(res)
    perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(res)})
    print(f"  [2/11] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(res):4d} results")

    print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
    t0 = perf_time.time()
    res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, state_under_test)
    dur = perf_time.time() - t0
    all_test_results.extend(res)
    perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(res)})
    print(f"  [3/11] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(res):4d} results")

    print(f"  Starting paymentPending Part 4 (detained)...")
    t0 = perf_time.time()
    res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze, M2_silver, H_silver, state_under_test)
    dur = perf_time.time() - t0
    all_test_results.extend(res)
    perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(res)})
    print(f"  [4/11] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(res):4d} results")

    print(f"  Starting appealSubmitted...")
    t0 = perf_time.time()
    all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, child_fields_to_exclude, M4_silver, M4_bronze))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [5/11] appealSubmitted".ljust(50) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [6/11] awaitingRespondentEvidence(a)".ljust(50) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting awaitingRespondentEvidence(b)...")
    t0 = perf_time.time()
    all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [7/11] awaitingRespondentEvidence(b)".ljust(50) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting listing...")
    t0 = perf_time.time()
    all_test_results.extend(run_listing(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bac, bll, b, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "listing", "test_from_state": "listing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [8/11] listing".ljust(50) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting prepareForHearing...")
    t0 = perf_time.time()
    all_test_results.extend(run_pfh(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "prepareForHearing", "test_from_state": "prepareForHearing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [9/11] prepareForHearing".ljust(50) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decision...")
    t0 = perf_time.time()
    all_test_results.extend(run_decision(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, child_fields_to_exclude))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decision", "test_from_state": "decision", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [10/11] decision".ljust(50) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

    print(f"  Starting decided(a)...")
    t0 = perf_time.time()
    all_test_results.extend(run_decidedA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
    dur = perf_time.time() - t0
    perf_timings.append({"test_name": "decided(a)", "test_from_state": "decided(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
    print(f"  [11/11] decided(a)".ljust(50) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results -- all complete")
    print(f"  Finished running all states.")

    print(f"Total tests collected: {len(all_test_results)}")

In [0]:
########################
# PROCESS RESULTS
########################
from collections import defaultdict
from Test_Functions.report_helpers import (
    count_by_status,
    create_run, create_results, create_perf_results,
    build_report_folder,
    write_csv, write_xlsx, write_html,
)

# ---- 1. COUNTS (visible, shared with helpers) ----
counts = count_by_status(all_test_results)

# ---- 2. CONSOLE SUMMARY (visible, easy to tweak) ----
grouped = defaultdict(list)
for r in all_test_results:
    grouped[r.test_from_state].append(r)

lines = []
lines.append("=" * 80)
lines.append(f"TEST EXECUTION REPORT -- {state_under_test}")
lines.append("-" * 80)
lines.append(f"Total tests : {counts['TOTAL']}")
lines.append(f"Passed      : {counts['PASS']}")
lines.append(f"Failed      : {counts['FAIL']}")
lines.append(f"No Data     : {counts['NO_DATA']}")
lines.append(f"Error       : {counts['ERROR']}")
lines.append("-" * 80)

for state in sorted(grouped):
    c = count_by_status(grouped[state])
    lines.append(f"\nSTATE: {state}")
    lines.append(f"Tests: {c['TOTAL']} | Passed: {c['PASS']} | Failed: {c['FAIL']} | No Data: {c['NO_DATA']} | Error: {c['ERROR']}")
    lines.append("-" * 60)
print("\n".join(lines))
# ---- 2b. PERFORMANCE TIMINGS ----
if perf_timings:
    lines2 = []
    lines2.append("")
    lines2.append("=" * 80)
    lines2.append("PERFORMANCE TIMINGS")
    lines2.append("-" * 80)
    total_secs = 0.0
    for pt in perf_timings:
        total_secs += pt["elapsed_seconds"]
        lines2.append(f"  {pt['test_name']:65s} {int(pt['elapsed_seconds']//3600)}h {int(pt['elapsed_seconds']%3600//60)}m {int(pt['elapsed_seconds']%60)}s ({pt['elapsed_seconds']:.1f}s)  {pt['result_count']} results")
    lines2.append("-" * 80)
    lines2.append(f"  {'TOTAL':65s} {int(total_secs//3600)}h {int(total_secs%3600//60)}m {int(total_secs%60)}s ({total_secs:.1f}s)")
    lines2.append("=" * 80)
    print("\n".join(lines2))


# ---- 3. IF THIS IS THE TOP STATE: WRITE FILES + TABLES ----
if state_under_test == this_notebook_state:
    # Build ONE folder for this run so all 3 files land together
    folder, timestamp, _ = build_report_folder(test_results_path, state_under_test)
    print(f"Reports folder: {folder}")

    try:
        csv_path = write_csv(all_test_results, state_under_test, folder, timestamp)
        print(f"CSV  : {csv_path}")
    except Exception as e:
        print(f"CSV  write failed: {e}")

    try:
        xlsx_path = write_xlsx(all_test_results, state_under_test, folder, timestamp)
        print(f"XLSX : {xlsx_path}")
    except Exception as e:
        print(f"XLSX write failed: {e}")

    try:
        html_path = write_html(all_test_results, state_under_test, folder, timestamp, counts=counts)
        print(f"HTML : {html_path}")
    except Exception as e:
        print(f"HTML write failed: {e}")

    # Spark tables
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(
            spark,
            run_user=run_user,
            run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name,
            run_tag=run_tag,
            run_status=run_status,
            state_under_test=state_under_test,
        )
        print(f"Created run_id={run_id}")
    except Exception as e:
        print(f"create_run FAILED: {e}")
        run_id = None

    if run_id:
        try:
            n = create_results(spark, run_id, all_test_results)
            print(f"Saved {n} result rows")
        except Exception as e:
            print(f"create_results FAILED: {e}")

        try:
            np = create_perf_results(spark, run_id, perf_timings, state_under_test)
            print(f"Saved {np} perf rows")
        except Exception as e:
            print(f"create_perf_results FAILED: {e}")

else:
    # Chain child — return data to parent
    import json
    from dataclasses import asdict
    all_test_results_string = json.dumps([asdict(r) for r in all_test_results])
    print(f"Exiting Notebook : {this_notebook_state} and returning Data to Parent notebook : {state_under_test}")
    dbutils.notebook.exit(all_test_results_string)